# DA-Fusion Pipeline

Drop-in replacement for `pipeline.ipynb` that adds support for DA-Fusion augmented images.

## Experiment structure
We compare three conditions:
- **Baseline**: Real data only (your existing pipeline)
- **RandAugment**: Real data + classical augmentation applied at feature extraction time
- **DA-Fusion**: Real data + diffusion-generated synthetic images


In [2]:
import numpy as np
import pandas as pd
import os
import sys
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor 
from sklearn.model_selection import train_test_split
import torch

# Add src/models to path so we can import existing modules
sys.path.insert(0, os.path.abspath('../models'))
import resnet as features
import data_utils

TRAIN_CSV          = '../../data/tabular/train/train.csv'
AUG_CSV            = '../../data/tabular/train/train_augmented.csv'  # from generate_augmented.py
TEST_CSV           = '../../data/tabular/test/test.csv'
IMAGE_FOLDER       = '../../data/image'
TARGETS            = data_utils.TARGETS
RANDOM_STATE       = 42

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cpu


## Helper: RandAugment baseline

Classical augmentation applied during feature extraction, acts as our comparison baseline against DA-Fusion.

In [3]:
from torchvision import transforms
from PIL import Image
import torch.nn as nn
import torchvision.models as models

class ResNetExtractorRandAugment:
    """
    Same as ResNetExtractor but applies RandAugment during feature extraction.
    For training images only — test images still get standard preprocessing.
    """
    def __init__(self, device='cpu', num_ops=2, magnitude=9):
        self.device = device
        resnet = models.resnet50(pretrained=True)
        self.model = nn.Sequential(*list(resnet.children())[:-1])
        self.model.to(device).eval()

        # Standard preprocessing (for test / no augmentation)
        self.preprocess_standard = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

        # RandAugment preprocessing (for training images)
        self.preprocess_augmented = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandAugment(num_ops=num_ops, magnitude=magnitude),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

    def get_features(self, image_path, augment=False):
        if not os.path.exists(image_path):
            return np.zeros(2048)
        try:
            img = Image.open(image_path).convert('RGB')
            img_left  = img.crop((0, 0, 1000, 1000))
            img_right = img.crop((1000, 0, 2000, 1000))
            preprocess = self.preprocess_augmented if augment else self.preprocess_standard
            t_left  = preprocess(img_left).unsqueeze(0).to(self.device)
            t_right = preprocess(img_right).unsqueeze(0).to(self.device)
            with torch.no_grad():
                f_left  = self.model(t_left).squeeze().cpu().numpy()
                f_right = self.model(t_right).squeeze().cpu().numpy()
            return (f_left + f_right) / 2.0
        except Exception as e:
            print(f'Error: {e}')
            return np.zeros(2048)

## Helper: Load augmented CSV

The augmented CSV (wide format) is produced by `generate_augmented.py`.
This function loads it and returns X features + y targets ready for training.

In [4]:
def extract_features_from_wide_csv(df_wide, image_folder, extractor,
                                   augment_synthetic=False):
    """
    Given a wide-format DataFrame (one row per image, targets as columns),
    extract ResNet50 features for each image.

    For augmented images (is_synthetic=True), the path is relative to image_folder/
    directly (since they're saved in data/image/augmented/).

    Args:
        augment_synthetic: If True, apply RandAugment to real images only
                           (synthetic images are already augmented by diffusion)
    """
    feats = []
    for idx, row in df_wide.iterrows():
        if idx % 100 == 0:
            print(f'  Processed {idx}/{len(df_wide)}...')

        rel_path = row['image_path']
        full_path = os.path.join(image_folder, rel_path)

        is_synthetic = row.get('is_synthetic', False)

        # Real images: optionally apply RandAugment
        # Synthetic images: already augmented, use standard preprocessing
        apply_aug = augment_synthetic and not is_synthetic
        feats.append(extractor.get_features(full_path, augment=apply_aug))

    return np.array(feats)


def load_augmented_csv(aug_csv_path):
    """
    Load the wide-format augmented CSV produced by generate_augmented.py.
    Returns (df, y) where y is shape (N, 5).
    """
    df = pd.read_csv(aug_csv_path)
    y = df[TARGETS].values
    return df, y

## Experiment 1: Baseline (real data only)
This is our reference score.

In [5]:
print('=== EXPERIMENT 1: Baseline (real data only) ===')

print('Loading training data...')
df_train, y_train = data_utils.load_train_data(TRAIN_CSV)
print(f'  {len(df_train)} images')

extractor_base = features.ResNetExtractor(device=device)

print('Extracting features...')
train_feats_base = []
for idx, path in enumerate(df_train['image_path']):
    if idx % 50 == 0: print(f'  {idx}/{len(df_train)}')
    train_feats_base.append(
        extractor_base.get_features(os.path.join(IMAGE_FOLDER, path))
    )
X_base = np.array(train_feats_base)

X_tr, X_val, y_tr, y_val = train_test_split(X_base, y_train,
                                              test_size=0.2,
                                              random_state=RANDOM_STATE)

rf_base = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_base.fit(X_tr, y_tr)
preds_base = rf_base.predict(X_val)

r2_base = data_utils.weighted_global_r2(y_val, preds_base)
rmse_base = data_utils.rmse_per_target(y_val, preds_base)

print(f'\n  Weighted R²: {r2_base:.4f}')
print(f'  RMSE per target: {rmse_base}')

=== EXPERIMENT 1: Baseline (real data only) ===
Loading training data...
  357 images
Loading ResNet50 on cpu...


c:\Users\Eier\anaconda3\envs\INF265\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Eier\anaconda3\envs\INF265\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Extracting features...
  0/357
  50/357
  100/357
  150/357
  200/357
  250/357
  300/357
  350/357

  Weighted R²: 0.6273
  RMSE per target: {'Dry_Green_g': 15.293800962663214, 'Dry_Dead_g': 10.285900299714848, 'Dry_Clover_g': 11.420730774918468, 'GDM_g': 14.394244265674601, 'Dry_Total_g': 17.51574440421652}


## Experiment 2: RandAugment baseline

Same real data, but with classical augmentation applied during feature extraction.

In [ ]:
print('=== EXPERIMENT 2: RandAugment baseline ===')

extractor_ra = ResNetExtractorRandAugment(device=device)

print('Extracting features with RandAugment...')
train_feats_ra = []
for idx, path in enumerate(df_train['image_path']):
    if idx % 50 == 0: print(f'  {idx}/{len(df_train)}')
    train_feats_ra.append(
        extractor_ra.get_features(os.path.join(IMAGE_FOLDER, path), augment=True)
    )
X_ra = np.array(train_feats_ra)

# Use same train/val split as baseline
idx_all = np.arange(len(X_ra))
idx_tr, idx_val = train_test_split(idx_all, test_size=0.2, random_state=RANDOM_STATE)

rf_ra = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_ra.fit(X_ra[idx_tr], y_train[idx_tr])
preds_ra = rf_ra.predict(X_ra[idx_val])

r2_ra = data_utils.weighted_global_r2(y_train[idx_val], preds_ra)
rmse_ra = data_utils.rmse_per_target(y_train[idx_val], preds_ra)

print(f'\n  Weighted R²: {r2_ra:.4f}')
print(f'  RMSE per target: {rmse_ra}')

=== EXPERIMENT 2: RandAugment baseline ===


c:\Users\Eier\anaconda3\envs\INF265\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Eier\anaconda3\envs\INF265\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Extracting features with RandAugment...
  0/357
  50/357
  100/357
  150/357
  200/357
  250/357
  300/357
  350/357

  Weighted R²: 0.5193
  RMSE per target: {'Dry_Green_g': 17.068372956179918, 'Dry_Dead_g': 10.7258972860536, 'Dry_Clover_g': 12.025874331139162, 'GDM_g': 17.062655537405615, 'Dry_Total_g': 19.76649646052507}


## Experiment 3: DA-Fusion

Train on real + synthetic images. Validate on the same held-out real images as the baseline.

**Requires**: `train_augmented.csv` and `data/image/augmented/` folder from `generate_augmented.py`.

In [ ]:
AUG_CSV_EXISTS = os.path.exists(AUG_CSV)

if not AUG_CSV_EXISTS:
    print(f'[SKIP] Augmented CSV not found at {AUG_CSV}')
    print('Run generate_augmented.py on Colab/HubroHub first, then re-run this cell.')
else:
    print('=== EXPERIMENT 3: DA-Fusion (real + synthetic) ===')

    df_aug, y_aug = load_augmented_csv(AUG_CSV)

    # Separate real vs synthetic rows
    real_mask = ~df_aug['is_synthetic'].astype(bool)
    df_real_aug = df_aug[real_mask].reset_index(drop=True)
    df_synth    = df_aug[~real_mask].reset_index(drop=True)
    y_real_aug  = y_aug[real_mask]
    y_synth     = y_aug[~real_mask]

    print(f'  Real images:      {len(df_real_aug)}')
    print(f'  Synthetic images: {len(df_synth)}')

    extractor_aug = features.ResNetExtractor(device=device)

    # Step 1: Extract features for real images, then split into train/val
    print('\nExtracting features for real images...')
    real_feats = []
    for idx, row in df_real_aug.iterrows():
        if idx % 50 == 0: print(f'  {idx}/{len(df_real_aug)}')
        real_feats.append(
            extractor_aug.get_features(os.path.join(IMAGE_FOLDER, row['image_path']))
        )
    X_real = np.array(real_feats)

    # Train/val split on real images only
    idx_all = np.arange(len(X_real))
    idx_tr_real, idx_val_real = train_test_split(idx_all, test_size=0.2,
                                                  random_state=RANDOM_STATE)

    X_val_daf  = X_real[idx_val_real]
    y_val_daf  = y_real_aug[idx_val_real]
    X_tr_real  = X_real[idx_tr_real]
    y_tr_real  = y_real_aug[idx_tr_real]

    # Step 2: Extract features for synthetic images (training only)
    print('\nExtracting features for synthetic images...')
    synth_feats = []
    for idx, row in df_synth.iterrows():
        if idx % 100 == 0: print(f'  {idx}/{len(df_synth)}')
        synth_feats.append(
            extractor_aug.get_features(os.path.join(IMAGE_FOLDER, row['image_path']))
        )
    X_synth = np.array(synth_feats)

    # Step 3: Combine real train + synthetic for training
    X_tr_daf = np.vstack([X_tr_real, X_synth])
    y_tr_daf = np.vstack([y_tr_real, y_synth])

    print(f'\n  Training set: {len(X_tr_daf)} images ({len(X_tr_real)} real + {len(X_synth)} synthetic)')
    print(f'  Validation set: {len(X_val_daf)} real images')

    # Step 4: Train and evaluate 
    rf_daf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
    rf_daf.fit(X_tr_daf, y_tr_daf)
    preds_daf = rf_daf.predict(X_val_daf)

    r2_daf = data_utils.weighted_global_r2(y_val_daf, preds_daf)
    rmse_daf = data_utils.rmse_per_target(y_val_daf, preds_daf)

    print(f'\n  Weighted R²: {r2_daf:.4f}')
    print(f'  RMSE per target: {rmse_daf}')

=== EXPERIMENT 3: DA-Fusion (real + synthetic) ===
  Real images:      357
  Synthetic images: 1071
Loading ResNet50 on cpu...


c:\Users\Eier\anaconda3\envs\INF265\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Eier\anaconda3\envs\INF265\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Extracting features for real images...
  0/357
  50/357
  100/357
  150/357
  200/357
  250/357
  300/357
  350/357

Extracting features for synthetic images...
  0/1071
  100/1071
  200/1071
  300/1071
  400/1071
  500/1071
  600/1071
  700/1071
  800/1071
  900/1071
  1000/1071

  Training set: 1356 images (285 real + 1071 synthetic)
  Validation set: 72 real images

  Weighted R²: 0.6202
  RMSE per target: {'Dry_Green_g': 15.133054584616517, 'Dry_Dead_g': 10.106846879158532, 'Dry_Clover_g': 11.537599079117404, 'GDM_g': 14.476912178333228, 'Dry_Total_g': 17.516946080994824}


## Results Summary

In [9]:
import pandas as pd

results = {
    'Baseline (real only)': {'R²': r2_base, **{f'RMSE {k}': v for k,v in rmse_base.items()}},
    'RandAugment':          {'R²': r2_ra,   **{f'RMSE {k}': v for k,v in rmse_ra.items()}},
}

if AUG_CSV_EXISTS:
    results['DA-Fusion'] = {'R²': r2_daf, **{f'RMSE {k}': v for k,v in rmse_daf.items()}}

df_results = pd.DataFrame(results).T.round(4)
print(df_results.to_string())

                          R²  RMSE Dry_Green_g  RMSE Dry_Dead_g  RMSE Dry_Clover_g  RMSE GDM_g  RMSE Dry_Total_g
Baseline (real only)  0.6273           15.2938          10.2859            11.4207     14.3942           17.5157
RandAugment           0.5193           17.0684          10.7259            12.0259     17.0627           19.7665
DA-Fusion             0.6202           15.1331          10.1068            11.5376     14.4769           17.5169
